In [5]:
#Import libraries
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [6]:
#Import the Dataset
data = pd.read_csv("Book_Recommend_Dataset.csv")

In [7]:
#Load the Dataset
data.head()

,asin,title,author,soldBy,imgUrl,productURL,stars,reviews,price,isKindleUnlimited,category_id,isBestSeller,isEditorsPick,isGoodReadsChoice,publishedDate,category_name
0,B00TZE87S4,Adult Children of Emotionally Immature Parents...,Lindsay C. Gibson,Amazon.com Services LLC,https://m.media-amazon.com/images/I/713KZTsaYp...,https://www.amazon.com/dp/B00TZE87S4,4.8,0,9.99,False,6,True,False,False,2015-06-01,Parenting & Relationships
1,B08WCKY8MB,"From Strength to Strength: Finding Success, Ha...",Arthur C. Brooks,Penguin Group (USA) LLC,https://m.media-amazon.com/images/I/A1LZcJFs9E...,https://www.amazon.com/dp/B08WCKY8MB,4.4,0,16.99,False,6,False,False,False,2022-02-15,Parenting & Relationships
2,B09KPS84CJ,Good Inside: A Guide to Becoming the Parent Yo...,Becky Kennedy,HarperCollins Publishers,https://m.media-amazon.com/images/I/71RIWM0sv6...,https://www.amazon.com/dp/B09KPS84CJ,4.8,0,16.99,False,6,False,True,False,2022-09-13,Parenting & Relationships
3,B07S7QPG6J,Everything I Know About Love: A Memoir,Dolly Alderton,HarperCollins Publishers,https://m.media-amazon.com/images/I/71QdQpTiKZ...,https://www.amazon.com/dp/B07S7QPG6J,4.2,0,9.95,True,6,False,True,False,2020-02-25,Parenting & Relationships
4,B00N6PEQV0,The Seven Principles for Making Marriage Work:...,John Gottman,Random House LLC,https://m.media-amazon.com/images/I/813o4WOs+w...,https://www.amazon.com/dp/B00N6PEQV0,4.7,0,13.99,False,6,False,False,False,2015-05-05,Parenting & Relationships


In [39]:
# Assuming your data is in a DataFrame called 'data'
# Extract year from 'publishedDate' column
data['year'] = pd.to_datetime(data['publishedDate']).dt.year

# Display the unique years
unique_years = data['year'].unique()
print(unique_years)

[2015. 2022. 2005. 2021.   nan 2023. 2017. 2019. 2020. 2008. 2014.]


##### Data Preprocessing

In [41]:
data['year'] = pd.to_numeric(data['year'],errors='coerce')

In [42]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 28 entries, 0 to 78886
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   title          28 non-null     object 
 1   author         28 non-null     object 
 2   soldBy         27 non-null     object 
 3   publishedDate  27 non-null     object 
 4   imgUrl         28 non-null     object 
 5   year           27 non-null     float64
 6   Metadata       27 non-null     object 
dtypes: float64(1), object(6)
memory usage: 1.8+ KB


In [43]:
#Deop unused Columns
data = data.dropna(subset=['title','author','soldBy','year'])

In [44]:
#Remove Duplicates
data.drop_duplicates(subset=['title'],inplace=True)

C:\Users\User\AppData\Local\Temp\ipykernel_17952\2069476085.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.drop_duplicates(subset=['title'],inplace=True)


In [45]:
data.head()

,title,author,soldBy,publishedDate,imgUrl,year,Metadata
0,Adult Children of Emotionally Immature Parents...,Lindsay C. Gibson,Amazon.com Services LLC,2015-06-01,https://m.media-amazon.com/images/I/713KZTsaYp...,2015.0,Adult Children of Emotionally Immature Parents...
1,"From Strength to Strength: Finding Success, Ha...",Arthur C. Brooks,Penguin Group (USA) LLC,2022-02-15,https://m.media-amazon.com/images/I/A1LZcJFs9E...,2022.0,"From Strength to Strength: Finding Success, Ha..."
2,Good Inside: A Guide to Becoming the Parent Yo...,Becky Kennedy,HarperCollins Publishers,2022-09-13,https://m.media-amazon.com/images/I/71RIWM0sv6...,2022.0,Good Inside: A Guide to Becoming the Parent Yo...
4,The Seven Principles for Making Marriage Work:...,John Gottman,Random House LLC,2015-05-05,https://m.media-amazon.com/images/I/813o4WOs+w...,2015.0,The Seven Principles for Making Marriage Work:...
5,The Glass Castle: A Memoir,Jeannette Walls,Simon and Schuster Digital Sales Inc,2005-03-01,https://m.media-amazon.com/images/I/71td5GDUZM...,2005.0,The Glass Castle: A MemoirJeannette WallsSimon...


In [22]:
#Combination of colums
data['Metadata'] = data['title'] + data['author'] + data['soldBy']

In [23]:
#Reload the Dataset
data.head()

,title,author,soldBy,publishedDate,imgUrl,year,Metadata
0,Adult Children of Emotionally Immature Parents...,Lindsay C. Gibson,Amazon.com Services LLC,2015-06-01,https://m.media-amazon.com/images/I/713KZTsaYp...,2015.0,Adult Children of Emotionally Immature Parents...
1,"From Strength to Strength: Finding Success, Ha...",Arthur C. Brooks,Penguin Group (USA) LLC,2022-02-15,https://m.media-amazon.com/images/I/A1LZcJFs9E...,2022.0,"From Strength to Strength: Finding Success, Ha..."
2,Good Inside: A Guide to Becoming the Parent Yo...,Becky Kennedy,HarperCollins Publishers,2022-09-13,https://m.media-amazon.com/images/I/71RIWM0sv6...,2022.0,Good Inside: A Guide to Becoming the Parent Yo...
4,The Seven Principles for Making Marriage Work:...,John Gottman,Random House LLC,2015-05-05,https://m.media-amazon.com/images/I/813o4WOs+w...,2015.0,The Seven Principles for Making Marriage Work:...
5,The Glass Castle: A Memoir,Jeannette Walls,Simon and Schuster Digital Sales Inc,2005-03-01,https://m.media-amazon.com/images/I/71td5GDUZM...,2005.0,The Glass Castle: A MemoirJeannette WallsSimon...


##### Feature Engineering

In [46]:
#Preprocessed the data
preprocessor = ColumnTransformer(
    transformers = [
        ("tfidf",TfidfVectorizer(stop_words='english'),'Metadata'),
        ('year_scaler',MinMaxScaler(),['year'])
    ]
)

In [51]:
#Create A Pipeline
pipeline = Pipeline([
    ("preprocessor",preprocessor),
    ("model",NearestNeighbors(n_neighbors=5,metric='cosine'))
])

In [52]:
pipeline.fit(data)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('tfidf', ...), ('year_scaler', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [53]:
#Save Model
import joblib
joblib.dump({'pipeline':pipeline,'data':data},'Book_Recommendation_Pipeline.joblib')

['Book_Recommendation_Pipeline.joblib']